In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
import yaml
import time
import json
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv

# 평가 지표 라이브러리
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 비전 검색 및 Gemini 통합
from byaldi import RAGMultiModalModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Langchain 및 프로젝트 모듈
import sys
sys.path.append('..')
from src import build_parent_retriever, run_corrective_agent, extract_target_company
from src.vision_utils import get_page_text_and_image
from langchain_classic.retrievers import (
    EnsembleRetriever,
    BM25Retriever,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


In [2]:
# API 및 검색기 로드
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

data_dir = "../data/raw/"
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))
base_retriever, all_docs = build_parent_retriever(pdf_files, force_rebuild=False, k=10)

# Method 1용 실시간 비전 DB 로드
index_name = "multi_doc_vision_index"
try:
    vision_retriever = RAGMultiModalModel.from_index(index_name)
    print(f"✅ 비전 DB '{index_name}' 로드 완료!")
except:
    print("⚠️ 비전 DB를 찾을 수 없습니다. (02 노트북 실행 필요)")
    vision_retriever = None

# Perplexity 평가용 언어 모델 로드
print("⏳ Perplexity 평가용 언어 모델 로드 중...")
try:
    ppl_tokenizer = AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")
    ppl_model = AutoModelForCausalLM.from_pretrained("skt/kogpt2-base-v2")
except:
    print("⚠️ PPL 모델 로드 실패. PPL 측정은 건너뜁니다.")
    ppl_model = None

nltk.download('punkt', quiet=True)


✅ Gemini API 키 로드 완료


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28174.40it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ 하드디스크에 저장된 DB와 문서를 불러옵니다!


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:35: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


🔍 BM25 키워드 검색기 복원 중...
✅ 하이브리드 검색기(캐시) 로드 완료!
⏳ Perplexity 평가용 언어 모델 로드 중...


Loading weights: 100%|██████████| 149/149 [00:00<00:00, 29800.74it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


True

In [3]:
import nltk
import ssl

# Mac/Windows 환경에서 SSL 인증서 에러로 다운로드가 조용히 막히는 것을 방지
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# 필수 토크나이저 데이터 명시적 다운로드 (최신 버전 호환을 위해 punkt_tab 추가)
nltk.download('punkt')
nltk.download('punkt_tab')

print("✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.")

✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# 🎯 5.4. 정성적 평가 및 5.2 성능 측정을 위한 Ground Truth(GT) 세트
eval_dataset = [
    {
        "query": "삼성SDI의 연결 현금흐름표에 기재된 2025년도의 당기순손익은 얼마인가요?",
        "gt_number": "-584,875", 
        "gt_text": "[파일명: [삼성SDI]사업보고서_2026.pdf, 연결 현금흐름표]에 따르면 2025년 당기순손익은 -584,875백만원입니다.", 
        "type": "표 단절 및 동명 기업 혼동"
    },
    {
        "query": "고려제약의 2025년 연구개발비용 총액은 얼마입니까?",
        "gt_number": "2,258,424",
        "gt_text": "[파일명: [고려제약]사업보고서_2026.pdf, 연구개발활동]에 따르면 2025년 연구개발비용 총액은 2,258,424천원입니다.",
        "type": "일반 재무 검색"
    },
    {
        "query": "삼성전자의 2024년 연결재무제표 기준 당기순이익은 얼마인가요?",
        "gt_number": "15,487,100", 
        "gt_text": "[파일명: [삼성전자]사업보고서_2026.pdf, 연결재무제표]에 따르면 2024년 당기순이익은 15,487,100백만원입니다.",
        "type": "일반 표 검색"
    },
    {
        "query": "LG에너지솔루션의 2025년 재무제표 주석에 기재된 '비용의 성격별 분류' 중 '종업원급여'는 얼마인가요?",
        "gt_number": "2,725,702", 
        "gt_text": "[파일명: [LG에너지솔루션]사업보고서_2026.pdf, 주석 (비용의 성격별 분류)]에 따르면 2025년 종업원급여는 2,725,702백만원입니다.",
        "type": "주석 내 특정 표 수치 검색"
    },
    {
        "query": "POSCO홀딩스의 2026년 이사회 의장 이름은 누구인가요?",
        "gt_number": "권태균",
        "gt_text": "[파일명: [POSCO홀딩스]사업보고서_2026.pdf, 이사회 등 회사의 기관에 관한 사항]에 따르면 2026년 이사회 의장은 권태균입니다.",
        "type": "비재무 및 인물 정보 검색"
    },
    {
        "query": "현대해상의 2025년 요약연결재무정보 상 영업수익은 얼마인가요?",
        "gt_number": "17,890,778,942", 
        "gt_text": "[파일명: [현대해상]사업보고서_2026.pdf, 연결포괄손익계산서]에 따르면 2025년 영업수익은 17,890,778,942천원입니다.",
        "type": "보험업 특화 항목 검색"
    },
    {
        "query": "한화손해보험의 요약재무정보에서 자본총계는 얼마로 기록되어 있나요?",
        "gt_number": "2,720,126", # ⚠️ 실제 PDF 확인 후 수정 요망
        "gt_text": "[파일명: [한화손해보험]사업보고서_2026.pdf, 요약재무정보]에 따르면 자본총계는 2,720,126백만원입니다.",
        "type": "보험업 요약재무정보 검색"
    },
    {
        "query": "한미반도체의 2025년 연구개발비 / 매출액 비율은 무엇인가요?",
        "gt_number": "4.3", # 비교 분석용 텍스트
        "gt_text": "[파일명: [한미반도체]사업보고서_2026.pdf, 연구개발비용]에 따르면 연구개발비 / 매출액 비율은 4.3%입니다.",
        "type": "표 내 수치 비교 및 분석"
    },
    {
        "query": "한화오션의 2025년 유형자산 장부금액을 연결재무정보에서 찾아주세요.",
        "gt_number": "5,273,173", 
        "gt_text": "[파일명: [한화오션]사업보고서_2026.pdf, 주석]에 따르면 2025년 유형자산 장부금액은 5,273,173백만원입니다.",
        "type": "주석 내 표 데이터 검색"
    },
    {
        "query": "카카오페이의 연결 현금흐름표에서 재무활동현금흐름은 얼마입니까?",
        "gt_number": "2,671,245,155", 
        "gt_text": "[파일명: [카카오페이]사업보고서_2026.pdf, 연결 현금흐름표]에 따르면 재무활동현금흐름은 2,671,245,155원입니다.",
        "type": "현금흐름표 긴 표 단절 테스트"
    }
]

In [5]:
def calc_exact_match(pred, gt_num):
    """Accuracy (Exact Match): 생성된 텍스트에 정답 숫자가 정확히 포함되어 있는지 확인"""
    # 숫자 포맷팅(콤마 등) 제거 후 비교
    clean_pred = pred.replace(",", "").replace(" ", "")
    clean_gt = gt_num.replace(",", "").replace(" ", "")
    return 1 if clean_gt in clean_pred else 0

def calc_rouge_bleu(pred, gt_text):
    """ROUGE & BLEU: 출처 및 맥락 일치도 평가"""
    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_score = scorer.score(gt_text, pred)['rougeL'].fmeasure
    
    # BLEU
    reference = [nltk.word_tokenize(gt_text)]
    candidate = nltk.word_tokenize(pred)
    smoothie = SmoothingFunction().method1
    bleu_score = sentence_bleu(reference, candidate, smoothing_function=smoothie)
    
    return rouge_score, bleu_score

def calc_perplexity(text):
    """Perplexity (PPL): 문장의 유창성 평가 (숫자가 많아 끊기면 PPL이 치솟음)"""
    if not ppl_model or not text.strip() or "찾지 못했습니다" in text:
        return np.nan
    
    encodings = ppl_tokenizer(text, return_tensors="pt")
    max_length = ppl_model.config.n_positions
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = ppl_model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss
        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
            
    return torch.exp(torch.stack(nlls).mean()).item()

def llm_as_a_judge(query, pred, gt_text):
    """LLM-as-a-judge: 환각, 억지 유추, 오답 여부를 프롬프트로 엄격히 평가"""
    # 텍스트가 비어있거나 실패 메시지면 무조건 1점(최하점) 부여
    if not pred.strip() or "실패" in pred or "찾지 못했" in pred:
        return 1.0
        
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
    prompt = f"""당신은 금융 RAG 시스템의 답변을 평가하는 심판입니다.
[질문]: {query}
[정답(Ground Truth)]: {gt_text}
[평가할 답변]: {pred}

위 3가지를 비교하여 '답변의 정확성과 무결성'을 1.0점에서 5.0점 사이의 소수점으로 평가하세요.
- 완벽한 정답과 출처: 5.0
- 정답은 맞으나 출처 불명확/부분 틀림: 3.0 ~ 4.0
- 환각, 틀린 수치, 엉뚱한 기업, 정답 없음: 1.0 ~ 2.0

[규칙]
반드시 평가 점수(숫자) 단 하나만 출력해야 합니다. 부가 설명, 단어, 기호는 절대 금지합니다.
출력 예시: 4.5
"""
    try:
        response = llm.invoke(prompt).content.strip()
        # 혹시 모를 문자 제거를 위해 숫자와 마침표만 남김
        import re
        score_str = re.sub(r'[^0-9.]', '', response)
        score = float(score_str)
        # 1.0 ~ 5.0 사이로 강제 조정
        return max(1.0, min(5.0, score))
    except Exception as e:
        print(f"Judge 에러: {e}")
        return 1.0


In [6]:
# --- 1. Method 0: Baseline RAG (텍스트 전용) ---
def run_method0_baseline(query):
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
    prompt = ChatPromptTemplate.from_template("다음 문서를 바탕으로 질문에 답하세요.\n\n문서:{context}\n질문:{input}")
    chain = create_retrieval_chain(base_retriever, create_stuff_documents_chain(llm, prompt))
    return chain.invoke({"input": query})["answer"]

# --- 2. Method 1: Simple Vision (진짜 비전 DB 이용) ---
def run_method1_vision(query):
    if not vision_retriever: return "비전 DB 로드 실패"
    results = vision_retriever.search(query, k=1)
    best_b64 = results[0].base64
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
    msg = HumanMessage(
        content=[
            {"type": "text", "text": f"공시 자료 이미지를 보고 질문에 답하세요.\n\n[질문]\n{query}"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{best_b64}"}}
        ]
    )
    return llm.invoke([msg]).content

# --- 3. Method 2: Dual-Path Basic (검증 에이전트 없는 하이브리드) ---
def run_method2_hybrid(query):
    # 텍스트와 상위 1개 이미지만 단순 결합
    retrieved = base_retriever.invoke(query)
    text_context = "\n".join([d.page_content for d in retrieved[:3]])
    
    # 첫 번째 결과의 이미지 가져오기
    first_doc = retrieved[0]
    _, img = get_page_text_and_image(first_doc.metadata['source'], first_doc.metadata['page'], all_docs)
    
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
    content = [{"type": "text", "text": f"문서 텍스트와 이미지를 참고하여 답변하세요.\n\n[텍스트]\n{text_context[:2000]}\n\n[질문]\n{query}"}]
    if img: content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}})
    
    return llm.invoke([HumanMessage(content=content)]).content

# --- 4. Method 3: 제안 시스템 (Ablation 및 SOTA) ---
def run_method3_agent(query, use_prefilter=True, max_expansions=3):
    if use_prefilter:
        extracted = extract_target_company(query)
        if extracted != "ALL":
            docs = [d for d in all_docs if extracted in d.metadata.get('source', '')]
            bm25 = BM25Retriever.from_documents(docs) if docs else base_retriever.retrievers[0]
            active_retriever = EnsembleRetriever(retrievers=[bm25, base_retriever.retrievers[1]], weights=[0.5, 0.5])
        else: active_retriever = base_retriever
    else: active_retriever = base_retriever

    retrieved = active_retriever.invoke(query)
    for doc in retrieved:
        ans = run_corrective_agent(query, doc.metadata.get('source'), doc.metadata.get('page', 0), all_docs, max_expansions)
        if "WRONG" not in ans and "NOT_FOUND" not in ans: return ans
    return "정답 찾기 실패"


In [7]:
import time
results = []
test_dataset = eval_dataset

print("🚀 전체 평가 파이프라인 가동 (Method 0 -> 1 -> 2 -> 3 Ablations -> 3 SOTA)")
for idx, data in enumerate(test_dataset):
    q, gt_num, gt_text = data["query"], data["gt_number"], data["gt_text"]
    print(f"\n▶️ [평가 {idx+1}/{len(test_dataset)}] {q}")
    
    # 순서 보장을 위해 OrderedDict 역할의 dict 정의
    preds = {}
    preds["Method 0 (Baseline)"] = run_method0_baseline(q)
    preds["Method 1 (Vision Real)"] = run_method1_vision(q)
    preds["Method 2 (Dual Basic)"] = run_method2_hybrid(q)
    preds["Method 3 (No-Filter)"] = run_method3_agent(q, use_prefilter=False, max_expansions=3)
    preds["Method 3 (No-Window)"] = run_method3_agent(q, use_prefilter=True, max_expansions=0)
    preds["Method 3 (SOTA)"] = run_method3_agent(q, use_prefilter=True, max_expansions=3)
    
    for model_name, pred_text in preds.items():
        rouge, bleu = calc_rouge_bleu(pred_text, gt_text)
        results.append({
            "Query_ID": idx + 1,
            "Type": data["type"],
            "Model": model_name,
            "Answer": pred_text,
            "Exact_Match": calc_exact_match(pred_text, gt_num),
            "ROUGE-L": round(rouge, 3),
            "BLEU": round(bleu, 3),
            "PPL": calc_perplexity(pred_text),
            "LLM_Judge": llm_as_a_judge(q, pred_text, gt_text)
        })
        # Rate Limit 방지 (각 모델 호출 사이)
        time.sleep(1)
    
    # 질문 하나 끝날 때마다 넉넉히 대기
    print("⏳ 다음 질문 준비 중... (10초)")
    time.sleep(10)

df_eval = pd.DataFrame(results)
summary_df = df_eval.groupby("Model")[["Exact_Match", "ROUGE-L", "BLEU", "PPL", "LLM_Judge"]].mean()

desired_order = [
    "Method 0 (Baseline)", 
    "Method 1 (Vision Real)", 
    "Method 2 (Dual Basic)", 
    "Method 3 (No-Filter)", 
    "Method 3 (No-Window)", 
    "Method 3 (SOTA)"
]
summary_df = summary_df.reindex(desired_order).reset_index()

print("\n🏆 최종 평가 결과 요약")
display(summary_df)
df_eval.to_csv("../data/comprehensive_evaluation_results.csv", index=False, encoding="utf-8-sig")


🚀 전체 평가 파이프라인 가동 (Method 0 vs Method 1 vs Ablation vs Method 2)...

▶️ [평가 1/10] 삼성SDI의 연결 현금흐름표에 기재된 2025년도의 당기순손익은 얼마인가요?

🔄 [Agent Loop 1] 분석 중인 페이지: [132, 133, 134]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
❌ 결과: 기업명 불일치 (Wrong Document - [고려제약]사업보고서(2026.03.12).pdf)

🔄 [Agent Loop 1] 분석 중인 페이지: [236, 237, 238]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)

🔄 [Agent Loop 1] 분석 중인 페이지: [557, 558, 559]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)

🔄 [Agent Loop 1] 분석 중인 페이지: [19, 20, 21]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)

🔄 [Agent Loop 1] 분석 중인 페이지: [200, 201, 202]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
❌ 결과: 기업명 불일치 (Wrong Document - [한미반도체]사업보고서(2026.03.12).pdf)

🔄 [Agent Loop 1] 분석 중인 페이지: [115, 116, 117]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)

🔄 [Agent Loop 1] 분석 중인 페이지: [253, 254, 255]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)

🔄 [Agent Loop 1] 분석 중인 페이지: [136, 137, 138]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}